# 04 · Cold start (Chapter 8)

Source: the local `ch8_rebuild_resumable.ipynb` (5090), including the retrieval-index fix (cells 22–23).

| Phase | Output |
|---|---|
| 0 drug-disjoint parent graph | Table 8.1 (bucket sizes) |
| 1 class filter (total ≥ 4 → 211 classes) and line graphs | Table 8.2 |
| 2 E1.1–E1.5 | Tables 8.3, 8.4, 8.7 |
| 3 warm-start control E2.1–E2.4 | Table 8.5 |
| 4 frozen ChemBERTa nearest prototype | Table 8.6 |
| fix | E1.4 / E2.4 with a union-graph retrieval index (the rows the thesis reports) |

Every phase skips work already on disk, so re-running after a disconnect is safe.

**Reproducing the thesis drug split:** a fresh Phase 0 gives a different split (node order depends on Python's string-hash seed). Point `THESIS_PARENT_GRAPH` at the cached `graphs_v4/parent_graph.bin` from the thesis run; it is copied into the new output folder and nothing in the thesis folder is modified.

In [1]:
ON_COLAB  = False
REPO_DIR  = "/home/ppcrniai/Documents/ddi-linegraph"
DATA_ROOT = "/home/ppcrniai/Documents/ddi-linegraph/ddi_data"
WORK_ROOT = "/home/ppcrniai/Documents/ddi-linegraph/ddi_runs"
TEST_MODE = True
N_TEST_DRUGS = 800   # must match the test graph built in notebook 00

In [2]:
import os, sys
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.exists(os.path.join(REPO_DIR, "pyproject.toml")):
        raise SystemExit(f"Repo not found at {REPO_DIR}. Upload or clone it there first.")
    !bash {REPO_DIR}/setup/colab_setup.sh
# the editable install is only picked up by a fresh interpreter; make it importable now
if os.path.join(REPO_DIR, "src") not in sys.path:
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))

In [3]:
import json, gc
import numpy as np
import torch
import dgl
import warnings
warnings.filterwarnings("ignore", message="Dataloader CPU affinity opt is not enabled")

from ddi.paths import get_paths
paths = get_paths(DATA_ROOT, WORK_ROOT, local_bin_dir=("/content" if ON_COLAB else None))
paths.make_output_dirs()
paths.show()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| TEST_MODE:", TEST_MODE)

  data_root              /home/ppcrniai/Documents/ddi-linegraph/ddi_data
  work_root              /home/ppcrniai/Documents/ddi-linegraph/ddi_runs
  raw_data_dir           /home/ppcrniai/Documents/ddi-linegraph/ddi_data/data   (does not exist yet)
  drug_db                /home/ppcrniai/Documents/ddi-linegraph/ddi_data/data/Drug_META_DDIE.db   (does not exist yet)
  event_db               /home/ppcrniai/Documents/ddi-linegraph/ddi_data/data/event_META_DDIE_5foldLabel.db   (does not exist yet)
  graph_zip              /home/ppcrniai/Documents/ddi-linegraph/ddi_data/ddi_graph_and_linegraph.zip   (does not exist yet)
  graph_bin              /home/ppcrniai/Documents/ddi-linegraph/ddi_runs/ddi_graph_and_linegraph.bin   (does not exist yet)
  thesis_warm_ckpt_dir   /home/ppcrniai/Documents/ddi-linegraph/ddi_data/thesis_smart_fusion_whole_graph_checkpoints   (does not exist yet)
  thesis_hidden384_dir   /home/ppcrniai/Documents/ddi-linegraph/ddi_data/thesis_hidden384_experiment   (does not ex

/home/ppcrniai/Documents/ddi-linegraph/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from ddi import coldstart as CS
import shutil

RUN_TAG = f"test{N_TEST_DRUGS}" if TEST_MODE else "v4"
d = CS.ColdStartDirs(base=paths.coldstart_dir, tag=RUN_TAG)
THESIS_PARENT_GRAPH = os.path.join(paths.thesis_ch8_dir, "graphs_v4", "parent_graph.bin")   # PLACEHOLDER if elsewhere

CS.banner("PHASE 0")
if os.path.exists(d.parent):
    print("  cached")
elif TEST_MODE:
    test_drug = dgl.load_graphs(os.path.join(paths.test_graph_dir, f"test_drug_graph_noneg_{N_TEST_DRUGS}.bin"))[0][0]
    CS.phase0_from_drug_graph(d, test_drug)
elif os.path.exists(THESIS_PARENT_GRAPH):
    shutil.copy(THESIS_PARENT_GRAPH, d.parent)
    print("  copied thesis parent graph (thesis drug split)")
else:
    from ddi.data import download_raw_data
    from ddi.paths import RAW_DATA_GDRIVE_ID
    drug_db, event_db = download_raw_data(paths.raw_data_dir, RAW_DATA_GDRIVE_ID)
    print("  WARNING: fresh rebuild; the drug split will NOT match the thesis")
    CS.phase0_build_parent(d, drug_db, event_db)
CS.mark(d, "phase0")
CS.phase0_report(d)


PHASE 0
  saved /home/ppcrniai/Documents/ddi-linegraph/ddi_runs/coldstart/graphs_test800/parent_graph.bin
  drugs train/val/test: 560/120/120
    TRAIN       30,448  48.86%
    VAL          1,152   1.85%
    S2          14,203  22.79%
    S3           1,561   2.50%
    DISCARD     14,956  24.00%
  intersections (all 0): 0,0,0


In [5]:
CS.banner("PHASE 1")
meta = CS.phase1(d)
CS.mark(d, "phase1")


PHASE 1
  rule=total>=4  C=122
  POST-FILTER buckets (Table 8.1):
    TRAIN       30,439  48.86%
    VAL          1,152   1.85%
    S2          14,199  22.79%
    S3           1,560   2.50%
    DISCARD     14,951  24.00%
  building train ...
    nodes 30,439 edges 1,782,099 (0s)
    [mem] train 0.8 GB
  building val ...
    nodes 1,152 edges 11,818 (0s)
    [mem] val 0.8 GB
  building S2_setupA ...
    nodes 44,638 edges 2,615,384 (1s)
    [mem] S2_setupA 0.8 GB
  building S2_setupB ...
    nodes 44,638 edges 3,126,651 (1s)
    [mem] S2_setupB 0.8 GB
  building S3_setupA ...
    nodes 31,999 edges 1,782,099 (0s)
    [mem] S3_setupA 0.8 GB
  building S3_setupB ...
    nodes 31,999 edges 1,802,136 (0s)
    [mem] S3_setupB 0.8 GB

  TABLE 8.2:
    train       nodes     30,439 edges      1,782,099
    val         nodes      1,152 edges         11,818
    S2_setupA   nodes     44,638 edges      2,615,384
    S2_setupB   nodes     44,638 edges      3,126,651
    S3_setupA   nodes     31,999

In [6]:
EPOCHS = dict(max_epochs=2, min_epochs=1) if TEST_MODE else {}
CS.banner("PHASE 2 - cold-start runs E1.1-E1.5")
r2 = CS.phase2(d, device, **EPOCHS)


PHASE 2 - cold-start runs E1.1-E1.5

  >>> E1_1_gated_ce (cross_entropy)


  ep001 loss=6.4464 val_strict=0.0062 val_default=0.0161 best=0.0062@1 noimp=0/20 *


  ep002 loss=6.0188 val_strict=0.0070 val_default=0.0173 best=0.0062@1 noimp=1/20
    S2_setupA   acc=0.2677 strict=0.0096 (default 0.0108) tail15=0.0010
    S2_setupB   acc=0.3019 strict=0.0121 (default 0.0135) tail15=0.0069
    S3_setupA   acc=0.2045 strict=0.0056 (default 0.0121) tail15=0.0000
    S3_setupB   acc=0.2756 strict=0.0077 (default 0.0172) tail15=0.0000

  >>> E1_2_gated_ldam (ldam)


  ep001 loss=11.4607 val_strict=0.0104 val_default=0.0270 best=0.0104@1 noimp=0/20 *


  ep002 loss=6.2965 val_strict=0.0116 val_default=0.0301 best=0.0116@2 noimp=0/20 *
    S2_setupA   acc=0.3049 strict=0.0271 (default 0.0304) tail15=0.0000
    S2_setupB   acc=0.3758 strict=0.0281 (default 0.0314) tail15=0.0000
    S3_setupA   acc=0.1462 strict=0.0090 (default 0.0187) tail15=0.0000
    S3_setupB   acc=0.2404 strict=0.0115 (default 0.0250) tail15=0.0000

  >>> E1_3_set_ldam (ldam)


  ep001 loss=12.4303 val_strict=0.0094 val_default=0.0245 best=0.0094@1 noimp=0/20 *


  ep002 loss=6.4227 val_strict=0.0152 val_default=0.0395 best=0.0152@2 noimp=0/20 *
    S2_setupA   acc=0.3047 strict=0.0273 (default 0.0306) tail15=0.0000
    S2_setupB   acc=0.3871 strict=0.0325 (default 0.0364) tail15=0.0000
    S3_setupA   acc=0.1449 strict=0.0102 (default 0.0212) tail15=0.0000
    S3_setupB   acc=0.2583 strict=0.0153 (default 0.0340) tail15=0.0000

  >>> E1_4_set_ldam_proto (ldam)


  ep001 loss=12.2526 val_strict=0.0148 val_default=0.0383 best=0.0148@1 noimp=0/20 *


  ep002 loss=6.2214 val_strict=0.0160 val_default=0.0407 best=0.0160@2 noimp=0/20 *
    S2_setupA   acc=0.3331 strict=0.0356 (default 0.0398) tail15=0.0000
    S2_setupB   acc=0.4068 strict=0.0374 (default 0.0419) tail15=0.0000
    S3_setupA   acc=0.1519 strict=0.0138 (default 0.0280) tail15=0.0000
    S3_setupB   acc=0.2410 strict=0.0162 (default 0.0341) tail15=0.0000

  >>> E1_5_concat_ldam (ldam)


  ep001 loss=11.0018 val_strict=0.0112 val_default=0.0290 best=0.0112@1 noimp=0/20 *


  ep002 loss=5.9498 val_strict=0.0149 val_default=0.0386 best=0.0149@2 noimp=0/20 *
    S2_setupA   acc=0.3259 strict=0.0345 (default 0.0387) tail15=0.0000
    S2_setupB   acc=0.4171 strict=0.0406 (default 0.0455) tail15=0.0000
    S3_setupA   acc=0.1288 strict=0.0077 (default 0.0165) tail15=0.0000
    S3_setupB   acc=0.2301 strict=0.0128 (default 0.0279) tail15=0.0000

  saved /home/ppcrniai/Documents/ddi-linegraph/ddi_runs/coldstart/results_test800/results_phase2.json


In [7]:
CS.banner("PHASE 3 - warm-start control E2.1-E2.4")
r3 = CS.phase3(d, device, **EPOCHS)


PHASE 3 - warm-start control E2.1-E2.4
  control split train/val/test: 21,296/4,558/4,559

  >>> E2_1_gated_ce (cross_entropy)


  ep001 loss=6.3018 val_strict=0.0067 val_default=0.0085 best=0.0067@1 noimp=0/20 *


  ep002 loss=5.9787 val_strict=0.0097 val_default=0.0123 best=0.0097@2 noimp=0/20 *
    ctl_test    acc=0.3123 strict=0.0124 (default 0.0158) tail15=0.0033

  >>> E2_2_gated_ldam (ldam)


  ep001 loss=13.5923 val_strict=0.0116 val_default=0.0149 best=0.0116@1 noimp=0/20 *


  ep002 loss=7.3986 val_strict=0.0196 val_default=0.0252 best=0.0196@2 noimp=0/20 *
    ctl_test    acc=0.3749 strict=0.0239 (default 0.0307) tail15=0.0000

  >>> E2_3_set_ldam (ldam)


  ep001 loss=14.8463 val_strict=0.0101 val_default=0.0129 best=0.0101@1 noimp=0/20 *


  ep002 loss=7.8919 val_strict=0.0233 val_default=0.0299 best=0.0233@2 noimp=0/20 *
    ctl_test    acc=0.3825 strict=0.0254 (default 0.0326) tail15=0.0000

  >>> E2_4_set_ldam_proto (ldam)


  ep001 loss=15.1479 val_strict=0.0081 val_default=0.0104 best=0.0081@1 noimp=0/20 *


  ep002 loss=7.9676 val_strict=0.0202 val_default=0.0259 best=0.0202@2 noimp=0/20 *
    ctl_test    acc=0.3678 strict=0.0261 (default 0.0335) tail15=0.0000

  saved /home/ppcrniai/Documents/ddi-linegraph/ddi_runs/coldstart/results_test800/results_phase3.json


In [8]:
CS.banner("PHASE 4 - ChemBERTa floor")
r4 = CS.phase4(d, device)


PHASE 4 - ChemBERTa floor
  S2_setupA   acc=0.0385 strict=0.0510
  S2_setupB   acc=0.0385 strict=0.0510
  S3_setupA   acc=0.0122 strict=0.0113
  S3_setupB   acc=0.0122 strict=0.0113


## Retrieval-index fix (E1.4, E2.4)
Re-evaluates the saved E1.4 / E2.4 checkpoints with an index covering every node of the evaluation graph. `oob` must be 0.

In [9]:
CS.banner("FIX E1.4"); f14 = CS.fix_e1_4(d, device)
CS.banner("FIX E2.4"); f24 = CS.fix_e2_4(d, device)


FIX E1.4

  --- S2: building index on the union node set ---
    recomputing 30,439 training rows (cache absent or mismatched)
         2,048/30,439    6.7%  eta      0s
    computing 14,199 held-out rows
         2,048/14,199   14.4%  eta      0s
    rare nodes in union graph: 422 of 44,638
    S2_setupA   strict=0.0353 tail15=0.0000 acc=0.3305 | oob=0 augmented=1,133
    S2_setupB   strict=0.0372 tail15=0.0000 acc=0.4024 | oob=0 augmented=1,664

  --- S3: building index on the union node set ---
    recomputing 30,439 training rows (cache absent or mismatched)
         2,048/30,439    6.7%  eta      0s
    computing 1,560 held-out rows
         1,560/1,560  100.0%  eta      0s
    rare nodes in union graph: 252 of 31,999
    S3_setupA   strict=0.0137 tail15=0.0000 acc=0.1519 | oob=0 augmented=24
    S3_setupB   strict=0.0173 tail15=0.0000 acc=0.2481 | oob=0 augmented=42

FIX E2.4
    recomputing 21,296 training rows (cache absent or mismatched)
         2,048/21,296    9.6%  eta    

In [10]:
CS.banner("TABLES")
_ = CS.phase5_tables(d)
CS.mark(d, "phase5")
print("\nThesis Table 8.3, E1.3 (Setup A/B, CS1 then CS2): 0.251 0.334 0.055 0.079")


TABLES

TABLE 8.3 - cold-start macro-F1 (STRICT, all classes)
  run                                      val   ep   S2_setupA   S2_setupB   S3_setupA   S3_setupB
  E1_1_gated_ce                         0.0062    1      0.0096      0.0121      0.0056      0.0077
  E1_2_gated_ldam                       0.0116    2      0.0271      0.0281      0.0090      0.0115
  E1_3_set_ldam                         0.0152    2      0.0273      0.0325      0.0102      0.0153
  E1_4_set_ldam_proto                   0.0160    2      0.0356      0.0374      0.0138      0.0162
  E1_5_concat_ldam                      0.0149    2      0.0345      0.0406      0.0077      0.0128
  E1_4_set_ldam_proto (fixed index)     0.0160    2      0.0353      0.0372      0.0137      0.0173

TABLE 8.4 - tail-class F1 (threshold 15)
  E1_1_gated_ce                             0.0010      0.0069      0.0000      0.0000
  E1_2_gated_ldam                           0.0000      0.0000      0.0000      0.0000
  E1_3_set_ldam      